In [0]:
CREATE OR REPLACE TABLE SDA_sensor_data (
  measurement_id INT,
  measurement_value DECIMAL(10, 2),
  measurement_time STRING
);

INSERT INTO SDA_sensor_data
  VALUES
    (131233, 1109.51, '07/10/2022 09:00:00'),
    (135211, 1662.74, '07/10/2022 11:00:00'),
    (523542, 1246.24, '07/10/2022 13:15:00'),
    (143562, 1124.50, '07/11/2022 15:00:00'),
    (346462, 1234.14, '07/11/2022 16:45:00');

with ranked as (
  select
    to_date(measurement_time, 'MM/dd/yyyy HH:mm:ss') as measurement_date,
    measurement_value,
    measurement_id,
    measurement_time,
    rank() over (
        partition by to_date(measurement_time, 'MM/dd/yyyy HH:mm:ss')
        order by measurement_time
      ) as rank
  from
    sda_sensor_data
)
select
  concat(date_format(measurement_date, 'MM/dd/yyyy'), ' 00:00:00') as measurement_day,
  sum(
    case
      when rank % 2 = 1 then measurement_value
      else 0
    end
  ) as odd_day_sum,
  sum(
    case
      when rank % 2 = 0 then measurement_value
      else 0
    end
  ) as even_day_sum
from
  ranked
group by
  measurement_date